# 05 · Validação do DW e consultas de negócio

Antes de confiar no DW para análise, duas perguntas: validação de integridade dos dados automática e
varificação de consultas do negócio por amostras

In [1]:
# Configuracao inicial
import os
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")


def find_project_root():
    current = Path.cwd().resolve()
    for p in [current] + list(current.parents):
        if (p / "src").exists() and (p / "data").exists() and (p / "config").exists():
            return p
    raise RuntimeError("Raiz do projeto nao encontrada")


def project_path(*segments):
    return find_project_root().joinpath(*segments)


root = find_project_root()
os.chdir(root)
print(f"Diretorio de trabalho: {root}")


Diretorio de trabalho: C:\Users\user\Downloads\Códigos\olist-ecommerce-pipeline\Template


## Validação automática

`DWValidator` roda checagens como contagem mínima de linhas, chaves órfãs entre fato e dimensão, valores
fora de faixa, reconciliação com a staging e resume tudo em `passed`/`failures`. É o mesmo
validador que a DAG do Airflow chama antes de liberar o dashboard, teste validador das DAGs

In [2]:
from src.etl.db import get_engine
from src.etl.validations import DWValidator

engine = get_engine()
result = DWValidator(engine).run_all()
print("Checks executados:", result.checks_run)
print("Passou?", result.passed)
result.failures

Checks executados: 41
Passou? True


[]

## Receita mensal

Junta `fact_order_items` com `dim_date` pela chave de data, agrupar por ano/mês sem precisar extrair toda vez de um timestamp

In [3]:
pd.read_sql(
    """
    SELECT d.year, d.month, SUM(f.price + f.freight_value) AS receita
    FROM dw.fact_order_items f
    JOIN dw.dim_date d ON f.order_purchase_date_key = d.date_key
    GROUP BY d.year, d.month
    ORDER BY d.year, d.month
    """,
    engine,
)

,year,month,receita
0,2016,9,354.75
1,2016,10,56808.84
2,2016,12,19.62
3,2017,1,137188.49
4,2017,2,286280.62
5,2017,3,432048.59
6,2017,4,412422.24
7,2017,5,586190.95
8,2017,6,502963.04
9,2017,7,584971.62


## Top categorias

Agrupando com `dim_products`.
`GROUP BY` direto, sem lidar com os nomes em português nem com categorias sem tradução

In [4]:
pd.read_sql(
    """
    SELECT p.product_category_name_english AS categoria, SUM(f.price) AS receita
    FROM dw.fact_order_items f
    JOIN dw.dim_products p ON f.product_key = p.product_key
    GROUP BY categoria ORDER BY receita DESC LIMIT 10
    """,
    engine,
)

,categoria,receita
0,health_beauty,1258681.34
1,watches_gifts,1205005.68
2,bed_bath_table,1036988.68
3,sports_leisure,988048.97
4,computers_accessories,911954.32
5,furniture_decor,729762.49
6,cool_stuff,635290.85
7,housewares,632248.66
8,auto,592720.11
9,garden_tools,485256.46


## Tempo médio de entrega por estado

Usa `fact_orders` (granularidade de pedido, não do item) tempo de entrega é métrica do pedido não do item. O
`WHERE delivery_time_days IS NOT NULL` exclui pedidos ainda não entregues, `NULL` legítimo

In [5]:
pd.read_sql(
    """
    SELECT c.customer_state AS estado, AVG(f.delivery_time_days) AS dias_medio
    FROM dw.fact_orders f
    JOIN dw.dim_customers c ON f.customer_key = c.customer_key
    WHERE f.delivery_time_days IS NOT NULL
    GROUP BY estado ORDER BY dias_medio DESC
    """,
    engine,
)

,estado,dias_medio
0,RR,29.386829
1,AP,27.185224
2,AM,26.425862
3,AL,24.544005
4,PA,23.772970
5,MA,21.573138
6,SE,21.519701
7,CE,21.266575
8,AC,21.035250
9,PB,20.426615
